<a href="https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This contract is for the starter content-refresh snapshot used for the lane. The full-release notebooks can be used later when the lane needs longitudinal warehouse data.

In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

# GitHub repository
REPO_URL = "https://github.com/shivam25th/flyrank-1st.git"
REPO = Path("/content/flyrank-1st")

# Clone repository if it is not already present
if not REPO.exists():
    print("Cloning repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        check=True
    )
else:
    print("Repository already exists.")

# IMPORTANT: move into the repository
os.chdir(REPO)

print("Current directory:", os.getcwd())

# Check that the data actually exists
DATA_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"

print("CSV path:", DATA_PATH)
print("CSV exists:", DATA_PATH.exists())

assert DATA_PATH.exists(), f"CSV not found: {DATA_PATH}"

# Load data
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Repository already exists.
Current directory: /content/flyrank-1st
CSV path: /content/flyrank-1st/data/raw/content_refresh_anonymized.csv
CSV exists: True
Dataset shape: (30000, 44)


## 1. Unit of analysis + time window

**One row = one anonymized content item/page.** The starter file is a cross-sectional snapshot with 90-day aggregate search-performance fields; it does not contain a row-level report date, so I will not invent a date range. Fields such as `impressions_90d` describe the observation window represented by the dataset.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Find the repository root when running locally or in Colab.
REPO = Path.cwd()
while not (REPO / "data" / "raw" / "content_refresh_anonymized.csv").exists() and REPO != REPO.parent:
    REPO = REPO.parent

DATA_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"
assert DATA_PATH.exists(), f"Starter CSV not found at {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print("Repository:", REPO)
print("Shape:", df.shape)

print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Duplicate content_id rows:", int(df["content_id"].duplicated().sum()))
print("Columns:", df.shape[1])
print("90-day fields present:", [c for c in df.columns if "90d" in c])


Repository: /content/flyrank-1st
Shape: (30000, 44)
Rows: 30000
Unique content_id: 30000
Duplicate content_id rows: 0
Columns: 44
90-day fields present: ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']


## 2. Fields: feature / label / context / excluded

**Candidate features:** `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`.

**Label:** `is_declining_label`, derived from `trend_direction == "down"`.

**Context/grouping:** `client_id` can be used to understand or group client-level behavior, but it is not a predictive feature because the identifier itself has no useful semantic meaning.

**Excluded:** `trend_direction` and `trend_pct` because they define/reveal the target; `content_id` because it is an identifier; `client_id` because it is a grouping/context field rather than a content signal.

In [ ]:
candidate_features = [
    "content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "word_count"
]
label_fields = ["trend_direction", "trend_pct"]
context_fields = ["client_id"]
identifier_fields = ["content_id"]

print("Candidate features:", [c for c in candidate_features if c in df.columns])
print("Label-derived fields:", [c for c in label_fields if c in df.columns])
print("Context fields:", [c for c in context_fields if c in df.columns])
print("Identifier fields:", [c for c in identifier_fields if c in df.columns])


Candidate features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Label-derived fields: ['trend_direction', 'trend_pct']
Context fields: ['client_id']
Identifier fields: ['content_id']


## 3. Verify it with queries (grain, counts, missing values, windows)

The checks below verify row grain, label distribution and missingness for the planned feature vector. A contract statement is only accepted after a corresponding check.

In [ ]:
features = [
    "content_age_days", "days_since_last_update",
    "impressions_90d", "avg_position", "ctr", "word_count"
]
print("Duplicate content IDs:", int(df["content_id"].duplicated().sum()))
print("\nLabel distribution:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nMissing values in planned features:")
print(df[features].isna().sum())

print("\nSummary of key numeric fields:")
print(df[features].describe().T[["count", "min", "50%", "max"]])


Duplicate content IDs: 0

Label distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Missing values in planned features:
content_age_days             0
days_since_last_update       0
impressions_90d              0
avg_position                 0
ctr                          0
word_count                7699
dtype: int64

Summary of key numeric fields:
                          count   min      50%       max
content_age_days        30000.0  90.0   236.00     564.0
days_since_last_update  30000.0   1.0    20.00     373.0
impressions_90d         30000.0   1.0   731.00  517715.0
avg_position            30000.0   0.0    10.80     245.0
ctr                     30000.0   0.0     0.07     100.0
word_count              22301.0   8.0  2877.00    9546.0


## 4. Data limits

This starter snapshot cannot establish causality or future performance. It aggregates 90-day behavior rather than providing a full daily time series. It also contains anonymized identifiers, so client-specific business context is unavailable. A future longitudinal model must also prevent future-window information from entering the features and should consider client/grouped validation.

In [ ]:
# Explicitly show that the starter dataset does not contain a row-level date field.
date_like = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
print("Date/time-like columns in starter data:", date_like)
print("Observation window fields:", [c for c in df.columns if "90d" in c])


Date/time-like columns in starter data: ['days_since_last_update']
Observation window fields: ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']


## Self-check

- [x] Unit of analysis and snapshot/window interpretation are stated.
- [x] Feature/label/context/excluded buckets are stated.
- [x] Claims are backed by checks.
- [x] Limits are explicit.
- [ ] Run top-to-bottom and commit before submission.
